# Proyecto Integrador 2 · Parte III — Evaluación, híbrido, sustitutos y afinidades inusuales
## Tareas 19 y 20

**Pregunta de interés P2:** *¿Qué proveedores son sustitutos entre sí para una misma entidad o categoría, y qué afinidades entidad–proveedor son inusuales?*

**Qué hace este notebook**
1. **Evalúa** los recomendadores sobre el **test** (pares nuevos desde 2026-03-01): populares, contenido, vecindario (item–item y user–user), ALS y un **híbrido**.
2. Elige el peso del híbrido **sin mirar el test** (con tres cortes de validación dentro de train) y luego muestra cómo queda en el test.
3. Mide la **incertidumbre** de los resultados (intervalos por remuestreo, varias semillas de ALS), la **cobertura** de las recomendaciones y **por qué** user–user supera a item–item.
4. Usa los vectores de contenido y el grafo para hallar **proveedores sustitutos**.
5. Busca **afinidades entidad–proveedor inusuales**: pares donde el proveedor vende algo muy distinto a lo que compra el resto de esa entidad.

> Todo lo que sale aquí es una **señal descriptiva para revisar**, no una acusación. Una afinidad inusual puede tener una razón normal (compra urgente, rubro mixto, texto pobre).

## 0. Configuración

In [1]:
import os, json, time
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize

RUTA_P2 = os.environ.get("P2_DATOS", "/content/drive/MyDrive/proyecto_integrador_seace/p2")
WEB     = os.environ.get("P2_WEB",   "/content/drive/MyDrive/proyecto_integrador_seace/web_artefactos/p2")
os.makedirs(WEB, exist_ok=True)

KS      = [5, 10, 20]               # se mide precisión y recall en estos top-k
ALFAS   = [round(a, 1) for a in np.arange(0, 1.01, 0.1)]   # peso de ALS en el híbrido (el resto es contenido)
PCT_VAL = 0.80                      # corte de validación principal (igual que el notebook 5)
PCT_VALS = [0.70, 0.80, 0.90]       # para elegir α se usan tres cortes de validación, no uno
N_BOOT = 1000                       # remuestreos para los intervalos de confianza
SEMILLAS_ALS = [42, 1, 2, 3, 4]     # semillas para medir cuánto varía ALS
PCT_INUSUAL = 2                     # "inusual" = entre el 2% de pares con menos afinidad
N_SUSTITUTOS_PROV, N_VECINOS_SUST = 150, 5
SEED = 42

train = pd.read_parquet(f"{RUTA_P2}/interacciones_train")
test  = pd.read_parquet(f"{RUTA_P2}/interacciones_test")
dim_prov = pd.read_parquet(f"{RUTA_P2}/proveedor_dim")

def nombre_visible(ruc, nombre):   # datos personales: persona natural → RUC enmascarado en vez de nombre
    r = str(ruc); return f"Persona natural (RUC {r[:5]}****{r[-2:]})" if r[:2] in ("10", "15", "16", "17") else nombre
dim_prov["nombre"] = [nombre_visible(r, n) for r, n in zip(dim_prov.ruc, dim_prov.nombre)]
dim_prov = dim_prov.set_index("proveedor_idx")
dim_ent  = pd.read_parquet(f"{RUTA_P2}/entidad_dim").set_index("entidad_idx")
palabras = pd.read_parquet(f"{RUTA_P2}/proveedores_palabras.parquet").set_index("proveedor_idx").palabras_clave

ent_ids  = np.sort(train.entidad_idx.unique()); prov_ids = np.sort(train.proveedor_idx.unique())
pos_ent  = {e: i for i, e in enumerate(ent_ids)}; pos_prov = {p: i for i, p in enumerate(prov_ids)}
NE, NP = len(ent_ids), len(prov_ids)
train["e"] = train.entidad_idx.map(pos_ent); train["p"] = train.proveedor_idx.map(pos_prov)

# Test: solo pares cuya entidad y proveedor están en el universo y que NO existían en train (lo "nuevo")
test = test.astype({"entidad_idx": int, "proveedor_idx": int})
n0 = len(test)
test["e"] = test.entidad_idx.map(pos_ent); test["p"] = test.proveedor_idx.map(pos_prov)
arranque = {"pares_test_total": int(n0),
            "pares_con_entidad_o_proveedor_sin_historial": int(test[["e", "p"]].isna().any(axis=1).sum()),
            "entidades_sin_historial": int(test.loc[test.e.isna(), "entidad_idx"].nunique()),
            "proveedores_sin_historial": int(test.loc[test.p.isna(), "proveedor_idx"].nunique())}
test = test.dropna(subset=["e", "p"]).astype({"e": int, "p": int})
n_conocidos = len(test)
test = test.merge(train[["e", "p"]].assign(ya=1), on=["e", "p"], how="left"); test = test[test.ya.isna()].drop(columns="ya")
arranque["pares_que_ya_existian_en_train"] = int(n_conocidos - len(test)); arranque["pares_evaluables"] = int(len(test))
print("arranque en frío (entidad o proveedor sin historial en train, no se pueden evaluar):", arranque)
print(f"pares del test: {n0} → {len(test)} evaluables (entidad y proveedor conocidos, y par nuevo)")
print("entidades con al menos un par nuevo en el test:", test.e.nunique())

def matriz(df):
    return csr_matrix((np.ones(len(df), dtype="float32"), (df.e.values, df.p.values)), shape=(NE, NP))
V = np.load(f"{RUTA_P2}/vectores_proveedor.npy")
assert (np.load(f"{RUTA_P2}/vectores_proveedor_ids.npy") == prov_ids).all()

arranque en frío (entidad o proveedor sin historial en train, no se pueden evaluar): {'pares_test_total': 10303, 'pares_con_entidad_o_proveedor_sin_historial': 3, 'entidades_sin_historial': 2, 'proveedores_sin_historial': 0, 'pares_que_ya_existian_en_train': 5374, 'pares_evaluables': 4926}
pares del test: 10303 → 4926 evaluables (entidad y proveedor conocidos, y par nuevo)
entidades con al menos un par nuevo en el test: 1331


## 1. Cómo se mide
Para cada entidad con pares nuevos en el test se ordenan los proveedores que **aún no tiene** y se mira el top-`k`:
- **Precisión@k:** de los `k` recomendados, ¿qué fracción contrató realmente? 
- **Recall@k:** de los proveedores nuevos que sí contrató, ¿qué fracción estaba en el top-`k`?

Los valores son bajos en términos absolutos porque entre ~9,800 proveedores se intenta adivinar con cuáles contratará una entidad en los próximos meses. Lo que importa es **cuánto mejor es cada modelo que el de referencia** (recomendar los más populares).

In [2]:
def metricas(S, vistos, df_eval, ks=KS, ents=None):
    """S: puntajes entidades×proveedores. vistos: matriz de lo que la entidad ya tiene. df_eval: pares nuevos a adivinar."""
    S = np.array(S, dtype="float32"); S[vistos.nonzero()] = -np.inf
    rel = df_eval.groupby("e").p.apply(set)
    if ents is not None: rel = rel[rel.index.isin(ents)]
    es = rel.index.values; kmax = max(ks)
    top = np.argpartition(-S[es], kmax, axis=1)[:, :kmax]
    top = np.take_along_axis(top, np.argsort(-np.take_along_axis(S[es], top, axis=1), axis=1), axis=1)
    out = {}
    for k in ks:
        pr, rc = [], []
        for i, e in enumerate(es):
            r = rel[e]; hit = np.array([p in r for p in top[i, :k]])
            pr.append(hit.sum() / k); rc.append(hit.sum() / len(r))
        out[k] = {"precision": np.mean(pr), "recall": np.mean(rc)}
    return out, len(es)

def z(S):   # normaliza cada fila (entidad): promedio 0 y desviación 1, para poder sumar puntajes de modelos distintos
    S = np.asarray(S, dtype="float32"); sd = S.std(1, keepdims=True); sd[sd == 0] = 1
    return (S - S.mean(1, keepdims=True)) / sd

def puntaje_contenido(R):    # perfil de la entidad = promedio de vectores de sus proveedores
    return normalize(R @ V) @ V.T

def similitud(R):
    co = (R.T @ R).toarray().astype("float32"); g = np.sqrt(np.diag(co)); g[g == 0] = 1
    S = co / g[:, None] / g[None, :]; np.fill_diagonal(S, 0); return S

def solo_k_vecinos(S, k):
    S2 = np.zeros_like(S); idx = np.argpartition(-S, k, axis=1)[:, :k]
    np.put_along_axis(S2, idx, np.take_along_axis(S, idx, axis=1), axis=1); return S2

## 2. Parámetros elegidos en el notebook 5 y modelos con train completo
Se leen los mejores parámetros que dejó la validación (`K` del vecindario y factores / regularización / alpha de ALS). Los factores de ALS entrenados con todo train ya están guardados.

In [3]:
vec_k = pd.read_csv(f"{WEB}/vecindario_k.csv"); als_p = pd.read_csv(f"{WEB}/als_parametros.csv")
K_BEST = int(vec_k.sort_values("recall@10", ascending=False).vecinos.iloc[0])
vec_u = pd.read_csv(f"{WEB}/vecindario_user_k.csv"); K_BEST_U = int(vec_u.sort_values("recall@10", ascending=False).vecinos.iloc[0])
mj = als_p.sort_values("recall@10", ascending=False).iloc[0]; RANK, REG = int(mj.factores), float(mj.regularizacion)
ALPHA = float(pd.read_csv(f"{WEB}/als_alpha.csv").sort_values("recall@10", ascending=False).alpha.iloc[0])
print("vecindario item–item K =", K_BEST, "| user–user K =", K_BEST_U, "| ALS:", RANK, "factores, regParam", REG, ", alpha", ALPHA)

R_full = matriz(train)
U = np.load(f"{RUTA_P2}/als_factores_entidad.npy"); I = np.load(f"{RUTA_P2}/als_factores_proveedor.npy")
S_als  = U @ I.T
S_cont = puntaje_contenido(R_full)
S_vec  = np.asarray(R_full @ solo_k_vecinos(similitud(R_full), K_BEST))                       # vecindario item–item
S_vecu = np.asarray(solo_k_vecinos(similitud(R_full.T.tocsr()), K_BEST_U) @ R_full)            # vecindario user–user
S_pop  = np.tile(np.asarray(R_full.sum(0)).ravel(), (NE, 1)).astype("float32")

vecindario item–item K = 800 | user–user K = 50 | ALS: 128 factores, regParam 1.0 , alpha 10.0


## 3. Elegir el peso del híbrido (sin mirar el test)
**Híbrido = α · ALS + (1 − α) · contenido**, con cada puntaje normalizado por entidad (promedio 0, desviación 1) para que se puedan sumar. `α = 0` es solo contenido; `α = 1` es solo ALS.

Para elegir `α` se usa la validación dentro de train: se entrenan ALS y contenido con la parte **más antigua** y se mide en la parte **más reciente**. Una sola validación puede engañar (el periodo reciente de train puede no parecerse al test), así que se repite con **tres cortes**: 70/30, 80/20 y 90/10. Se elige el `α` con mejor recall@10 **promedio** de los tres cortes. **El test no interviene en la elección.**

In [4]:
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
spark = (SparkSession.builder.master("local[2]").appName("P2_hibrido").config("spark.driver.memory", "3g")
         .config("spark.sql.shuffle.partitions", "8").config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

def entrenar_als(df, rank, reg, alpha=10.0, iteraciones=10, seed=SEED):
    sdf = spark.createDataFrame(pd.DataFrame({"e": df.e.astype(int), "p": df.p.astype(int), "valor": np.log1p(df.n_adjudicaciones).astype("float64")}))
    m = ALS(userCol="e", itemCol="p", ratingCol="valor", implicitPrefs=True, alpha=alpha, rank=rank, regParam=reg, maxIter=iteraciones, seed=seed).fit(sdf)
    Uu = np.zeros((NE, rank), dtype="float32"); Ii = np.zeros((NP, rank), dtype="float32")
    for r in m.userFactors.collect(): Uu[r.id] = r.features
    for r in m.itemFactors.collect(): Ii[r.id] = r.features
    return Uu, Ii


filas, cortes = [], {}
for pv in PCT_VALS:
    corte = train.fecha_primera.quantile(pv)
    tr_in = train[train.fecha_primera <= corte]; va = train[train.fecha_primera > corte]
    va = va[va.e.isin(tr_in.e) & va.p.isin(tr_in.p)]
    R_in = matriz(tr_in); cortes[pv] = (R_in, va)
    U_in, I_in = entrenar_als(tr_in, RANK, REG, alpha=ALPHA)
    Z_als_in, Z_cont_in = z(U_in @ I_in.T), z(puntaje_contenido(R_in))
    for a in ALFAS:
        m_val, _ = metricas(a * Z_als_in + (1 - a) * Z_cont_in, R_in, va, ks=[10])
        filas.append({"alfa": a, "corte": int(round(pv * 100)), "recall": m_val[10]["recall"], "pares_val": len(va)})
    print(f"corte {int(round(pv * 100))}/{int(round(100 - pv * 100))}: {len(tr_in)} pares para entrenar, {len(va)} para validar")
vc = pd.DataFrame(filas)
fila_test = []
for a in ALFAS:
    (m_te, _) = metricas(a * z(S_als) + (1 - a) * z(S_cont), R_full, test, ks=[10])
    fila_test.append({"alfa": a, "test_recall@10": round(m_te[10]["recall"], 4)})
alfa_df = pd.DataFrame({"alfa": ALFAS})
alfa_df["val_recall@10"] = vc.groupby("alfa").recall.mean().round(4).values
for pv in PCT_VALS:
    c_ = int(round(pv * 100)); alfa_df[f"val_recall@10_corte{c_}"] = vc[vc.corte == c_].recall.round(4).values
alfa_df = alfa_df.merge(pd.DataFrame(fila_test), on="alfa"); alfa_df.to_csv(f"{WEB}/hibrido_alfa.csv", index=False)
ALFA = float(alfa_df.sort_values("val_recall@10", ascending=False).alfa.iloc[0])
print(alfa_df.to_string(index=False))
print("\nα que elige cada corte (mejor recall@10):", {int(round(pv * 100)): float(vc[vc.corte == int(round(pv * 100))].sort_values("recall", ascending=False).alfa.iloc[0]) for pv in PCT_VALS})
print("α elegido con el promedio de los tres cortes:", ALFA)

corte 70/30: 30943 pares para entrenar, 11137 para validar


corte 80/20: 35366 pares para entrenar, 7820 para validar


corte 90/10: 39784 pares para entrenar, 4121 para validar


 alfa  val_recall@10  val_recall@10_corte70  val_recall@10_corte80  val_recall@10_corte90  test_recall@10
  0.0         0.1815                 0.1741                 0.1766                 0.1937          0.0799
  0.1         0.2032                 0.1966                 0.1953                 0.2176          0.1111
  0.2         0.1997                 0.1948                 0.1899                 0.2145          0.1293
  0.3         0.1931                 0.1888                 0.1851                 0.2055          0.1390
  0.4         0.1831                 0.1780                 0.1777                 0.1936          0.1432
  0.5         0.1726                 0.1657                 0.1651                 0.1869          0.1476
  0.6         0.1635                 0.1529                 0.1571                 0.1806          0.1520
  0.7         0.1559                 0.1469                 0.1510                 0.1699          0.1484
  0.8         0.1489                 0.1398   

### ¿Por qué la validación y el test pueden preferir α distintos?
La validación toma los pares más recientes de **train**; el test, los pares desde 2026-03-01. Si los proveedores que aparecen en cada uno tienen **historias distintas** (más o menos entidades atrás), un modelo basado en historia (ALS) rinde distinto en cada uno. Se compara cuánta historia tienen los proveedores que hay que adivinar.

In [5]:
diag_val = {}
for nombre, (R_, df_) in [("validación (corte 80/20)", (cortes[0.80][0], cortes[0.80][1])), ("test", (R_full, test))]:
    g = np.asarray(R_.sum(0)).ravel()[df_.p.values]
    diag_val[nombre] = {"pares": int(len(df_)), "mediana_entidades_del_proveedor": float(np.median(g)),
                        "pct_proveedor_con_2_o_menos_entidades": round(100 * float(np.mean(g <= 2)), 1)}
print(pd.DataFrame(diag_val).T.to_string())

                           pares  mediana_entidades_del_proveedor  pct_proveedor_con_2_o_menos_entidades
validación (corte 80/20)  7820.0                              4.0                                   39.6
test                      4926.0                              6.0                                   18.6


## 4. Resultados en el test
Todos los modelos usan **todo train** y se miden sobre los pares nuevos del test. Además de "todas las entidades", se separan las entidades en tres grupos según **cuántos proveedores tenían en train** (pocas / medias / muchas): ahí se ve dónde ayuda cada modelo (el contenido debería ayudar más a las entidades con poca historia).

In [6]:
S_hib = ALFA * z(S_als) + (1 - ALFA) * z(S_cont)
modelos = {"populares (referencia)": S_pop, "contenido": S_cont, "vecindario item–item": S_vec, "vecindario user–user": S_vecu, "ALS": S_als, f"híbrido (α={ALFA})": S_hib}

grado = pd.Series(np.asarray(R_full.sum(1)).ravel(), index=np.arange(NE))
ev = test.e.unique(); q1, q2 = np.quantile(grado[ev], [1/3, 2/3]).astype(int)
segmentos = {"todas": ev,
             f"pocas (≤{q1} proveedores)": ev[grado[ev] <= q1],
             f"medias ({q1+1}–{q2})": ev[(grado[ev] > q1) & (grado[ev] <= q2)],
             f"muchas (>{q2})": ev[grado[ev] > q2]}
filas = []
for nom, S in modelos.items():
    for seg, ents in segmentos.items():
        m, n = metricas(S, R_full, test, ks=KS, ents=ents)
        for k in KS:
            if seg != "todas" and k != 10: continue
            filas.append({"modelo": nom, "segmento": seg, "k": k, "precision": round(m[k]["precision"], 4),
                          "recall": round(m[k]["recall"], 4), "n_entidades": n})
res = pd.DataFrame(filas); res.to_csv(f"{WEB}/eval_resultados.csv", index=False)
print(res[(res.segmento == "todas")].pivot(index="modelo", columns="k", values="recall").round(4).to_string())
print()
print(res[(res.k == 10) & (res.segmento != "todas")].pivot(index="modelo", columns="segmento", values="recall").round(4).to_string())

k                           5       10      20
modelo                                        
ALS                     0.0870  0.1356  0.2057
contenido               0.0534  0.0799  0.1219
híbrido (α=0.1)         0.0777  0.1111  0.1547
populares (referencia)  0.0129  0.0223  0.0421
vecindario item–item    0.0560  0.0830  0.1282
vecindario user–user    0.0822  0.1190  0.1809

segmento                medias (10–26)  muchas (>26)  pocas (≤9 proveedores)
modelo                                                                      
ALS                             0.1419        0.0908                  0.1708
contenido                       0.0830        0.0428                  0.1111
híbrido (α=0.1)                 0.1164        0.0562                  0.1565
populares (referencia)          0.0246        0.0265                  0.0166
vecindario item–item            0.0858        0.0636                  0.0982
vecindario user–user            0.1306        0.0808                  0.1437


## 4b. ¿Qué tan seguros son estos números?
Con 1,331 entidades, una diferencia de 0.01 puede ser ruido. Tres comprobaciones:
- **Intervalo de confianza del 95% (bootstrap):** se remuestrean las entidades con reemplazo `N_BOOT` veces y se recalcula el promedio. Si dos modelos tienen intervalos que se separan, la diferencia es creíble. Para comparar dos modelos se remuestrea **el mismo conjunto de entidades** y se mira el intervalo de la **diferencia**: si no incluye 0, los modelos son distinguibles.
- **Semillas de ALS:** ALS parte de números al azar; se entrena `len(SEMILLAS_ALS)` veces con semillas distintas para ver cuánto cambia el resultado.
- **Cobertura y popularidad:** cuántos proveedores distintos aparecen en las recomendaciones, y qué parte del top-10 son los 100 proveedores más populares. Un modelo que recomienda siempre lo mismo tiene cobertura baja (sesgo a popularidad).

In [7]:
def por_entidad(S, vistos, df_eval, k=10):
    S = np.array(S, dtype="float32"); S[vistos.nonzero()] = -np.inf
    rel = df_eval.groupby("e").p.apply(set); es = rel.index.values
    top = np.argpartition(-S[es], k, axis=1)[:, :k]
    top = np.take_along_axis(top, np.argsort(-np.take_along_axis(S[es], top, axis=1), axis=1), axis=1)
    rc = []
    for i, e in enumerate(es):
        r = rel[e]; hit = np.array([p in r for p in top[i]])
        rc.append(hit.sum() / len(r))
    return es, np.array(rc), top

pe = {nom: por_entidad(S, R_full, test) for nom, S in modelos.items()}
es0 = pe["ALS"][0]; n_ent = len(es0)
idx = np.random.RandomState(SEED).randint(0, n_ent, size=(N_BOOT, n_ent))     # mismas entidades remuestreadas para todos
pc = lambda v: (round(float(np.percentile(v, 2.5)), 4), round(float(np.percentile(v, 97.5)), 4))
filas = []
for nom, (e_, rc, top) in pe.items():
    assert (e_ == es0).all()
    r_lo, r_hi = pc(rc[idx].mean(1))
    filas.append({"modelo": nom, "recall@10": round(float(rc.mean()), 4), "recall_ic_bajo": r_lo, "recall_ic_alto": r_hi, "n_entidades": n_ent})
eval_ic = pd.DataFrame(filas); eval_ic.to_csv(f"{WEB}/eval_ic.csv", index=False)
print(eval_ic.to_string(index=False))

nom_h = f"híbrido (α={ALFA})"
comparaciones = [("vecindario user–user", "vecindario item–item"), ("ALS", "vecindario user–user"), ("ALS", nom_h),
                 (nom_h, "contenido"), ("vecindario item–item", "contenido"), ("ALS", "populares (referencia)")]
filas = []
for a_, b_ in comparaciones:
    dif = (pe[a_][1] - pe[b_][1])[idx].mean(1); lo, hi = pc(dif)
    filas.append({"comparacion": f"{a_} − {b_}", "dif_recall@10": round(float((pe[a_][1] - pe[b_][1]).mean()), 4), "ic_bajo": lo, "ic_alto": hi,
                  "distinguible": bool(lo > 0 or hi < 0)})
eval_dif = pd.DataFrame(filas); eval_dif.to_csv(f"{WEB}/eval_diferencias.csv", index=False)
print(); print(eval_dif.to_string(index=False))

pop100 = set(np.argsort(-np.asarray(R_full.sum(0)).ravel())[:100])
filas = []
for nom, (e_, rc, top) in pe.items():
    distintos = len(np.unique(top))
    filas.append({"modelo": nom, "proveedores_distintos": int(distintos), "cobertura_pct": round(100 * distintos / NP, 1),
                  "pct_recomendaciones_en_top100_populares": round(100 * float(np.isin(top, list(pop100)).mean()), 1)})
cobertura = pd.DataFrame(filas); cobertura.to_csv(f"{WEB}/cobertura.csv", index=False)
print(); print(cobertura.to_string(index=False))

                modelo  recall@10  recall_ic_bajo  recall_ic_alto  n_entidades
populares (referencia)     0.0223          0.0167          0.0287         1331
             contenido     0.0799          0.0683          0.0916         1331
  vecindario item–item     0.0830          0.0714          0.0945         1331
  vecindario user–user     0.1190          0.1050          0.1323         1331
                   ALS     0.1356          0.1206          0.1508         1331
       híbrido (α=0.1)     0.1111          0.0968          0.1242         1331

                                comparacion  dif_recall@10  ic_bajo  ic_alto  distinguible
vecindario user–user − vecindario item–item         0.0360   0.0235   0.0490          True
                 ALS − vecindario user–user         0.0165   0.0056   0.0280          True
                      ALS − híbrido (α=0.1)         0.0244   0.0128   0.0371          True
                híbrido (α=0.1) − contenido         0.0312   0.0234   0.0388      

In [8]:
filas = []
for sd in SEMILLAS_ALS:
    U_s, I_s = entrenar_als(train, RANK, REG, alpha=ALPHA, seed=sd)
    m_s, _ = metricas(U_s @ I_s.T, R_full, test, ks=[10])
    filas.append({"semilla": sd, "recall@10": round(m_s[10]["recall"], 4)})
    print("semilla", sd, "recall@10 =", filas[-1]["recall@10"])
als_sem = pd.DataFrame(filas); als_sem.to_csv(f"{WEB}/als_semillas.csv", index=False)
print("promedio:", als_sem["recall@10"].mean().round(4), "| desviación:", als_sem["recall@10"].std().round(4), "| mín–máx:", als_sem["recall@10"].min(), als_sem["recall@10"].max())
spark.stop()

semilla 42 recall@10 = 0.1356


semilla 1 recall@10 = 0.1369


semilla 2 recall@10 = 0.1341


semilla 3 recall@10 = 0.1393


semilla 4 recall@10 = 0.137
promedio: 0.1366 | desviación: 0.0019 | mín–máx: 0.1341 0.1393


## 4c. ¿Por qué user–user supera a item–item?
En clase se dijo que item–item suele funcionar mejor; aquí pasa al revés. Una explicación posible: la similitud entre dos **proveedores** se calcula con las pocas **entidades** a las que atiende cada uno (la mediana es muy baja), mientras que la similitud entre dos **entidades** se apoya en bastantes proveedores. Si es así, item–item debería quedar peor sobre todo cuando el proveedor a adivinar tiene **pocas entidades**. Se comprueba separando los pares nuevos del test según cuántas entidades tenía el proveedor en train.

In [9]:
g_ent = np.asarray(R_full.sum(1)).ravel(); g_prov = np.asarray(R_full.sum(0)).ravel()
dens = {"proveedores_por_entidad_mediana": float(np.median(g_ent)), "entidades_por_proveedor_mediana": float(np.median(g_prov)),
        "pct_proveedores_con_2_o_menos_entidades": round(100 * float(np.mean(g_prov <= 2)), 1),
        "pct_entidades_con_2_o_menos_proveedores": round(100 * float(np.mean(g_ent <= 2)), 1)}
print(dens)
grupos = {"≤2 entidades": g_prov <= 2, "3 a 9 entidades": (g_prov >= 3) & (g_prov <= 9), "10 o más entidades": g_prov >= 10}
filas = []
for gn, msk in grupos.items():
    sub = test[msk[test.p.values]]
    for nom in ["vecindario item–item", "vecindario user–user", "ALS"]:
        m_g, n_g = metricas(modelos[nom], R_full, sub, ks=[10])
        filas.append({"grupo_del_proveedor": gn, "modelo": nom, "recall@10": round(m_g[10]["recall"], 4), "pares_nuevos": int(len(sub)), "entidades": n_g})
diag_grado = pd.DataFrame(filas); diag_grado.to_csv(f"{WEB}/diagnostico_grado.csv", index=False)
print(diag_grado.pivot(index="grupo_del_proveedor", columns="modelo", values="recall@10").round(4).to_string())
print(diag_grado.groupby("grupo_del_proveedor").pares_nuevos.first().to_string())
with open(f"{WEB}/reporte_evaluacion.json", "w", encoding="utf-8") as f:
    json.dump({"arranque_en_frio": arranque, "validacion_vs_test": diag_val, "densidad": dens}, f, ensure_ascii=False, indent=1)

{'proveedores_por_entidad_mediana': 8.0, 'entidades_por_proveedor_mediana': 3.0, 'pct_proveedores_con_2_o_menos_entidades': 45.7, 'pct_entidades_con_2_o_menos_proveedores': 9.9}


modelo                  ALS  vecindario item–item  vecindario user–user
grupo_del_proveedor                                                    
10 o más entidades   0.2205                0.1097                0.2496
3 a 9 entidades      0.0928                0.0580                0.0512
≤2 entidades         0.0308                0.0675                0.0132
grupo_del_proveedor
10 o más entidades    1637
3 a 9 entidades       2374
≤2 entidades           915


## 5. Proveedores sustitutos
**Idea:** dos proveedores son **sustitutos** si venden cosas muy parecidas (mismo texto en los procesos que ganan) y a la vez **aparecen en los mismos lugares**: atienden a las mismas entidades o se presentan a los mismos procesos.

Para cada proveedor se toman sus vecinos más parecidos en **contenido** (los vectores del notebook 4) y se mira, para cada par:
- **Similitud de contenido** (coseno, 0 a 1).
- **Entidades en común:** a cuántas entidades les adjudicaron ambos (en train).
- **Procesos en común (compitieron):** a cuántos procesos se presentaron los dos como postores (arista del grafo del notebook 2).

Como referencia se compara contra pares tomados al azar: si los vecinos por contenido comparten entidad mucho más que el azar, la similitud de contenido sí identifica sustitutos reales.

In [10]:
sim = V @ V.T; np.fill_diagonal(sim, -1)
K10 = 10
vec10 = np.argpartition(-sim, K10, axis=1)[:, :K10]
vec10 = np.take_along_axis(vec10, np.argsort(-np.take_along_axis(sim, vec10, axis=1), axis=1), axis=1)
sim10 = np.take_along_axis(sim, vec10, axis=1); del sim

co_ent = lambda a, b: np.asarray(R_full[:, a].multiply(R_full[:, b]).sum(0)).ravel()
A = np.repeat(np.arange(NP), K10); B = vec10.ravel()
ent_comun = np.asarray((R_full.T[A].multiply(R_full.T[B])).sum(1)).ravel()       # entidades en común por par

aristas = pd.read_parquet(f"{RUTA_P2}/grafo_aristas_peso1")
aristas["a"] = aristas.prov_a.map(pos_prov); aristas["b"] = aristas.prov_b.map(pos_prov)
aristas = aristas.dropna(subset=["a", "b"]).astype({"a": int, "b": int})
aristas = pd.concat([aristas[["a", "b", "peso"]], aristas.rename(columns={"a": "b", "b": "a"})[["a", "b", "peso"]]])
pares = pd.DataFrame({"a": A, "b": B, "sim": sim10.ravel(), "ent_comun": ent_comun}).merge(aristas, on=["a", "b"], how="left")
pares["peso"] = pares.peso.fillna(0).astype(int)

# referencia: pares al azar
rs = np.random.RandomState(SEED); ra, rb = rs.randint(0, NP, 100_000), rs.randint(0, NP, 100_000)
az_ent = np.asarray((R_full.T[ra].multiply(R_full.T[rb])).sum(1)).ravel()
az = pd.DataFrame({"a": ra, "b": rb}).merge(aristas, on=["a", "b"], how="left")
resumen = {"pares_vecinos": int(len(pares)),
           "vecinos_comparten_entidad_pct": round(100 * float((pares.ent_comun > 0).mean()), 1),
           "azar_comparten_entidad_pct": round(100 * float((az_ent > 0).mean()), 2),
           "vecinos_compitieron_pct": round(100 * float((pares.peso > 0).mean()), 1),
           "azar_compitieron_pct": round(100 * float(az.peso.notna().mean()), 2),
           "similitud_mediana_vecinos": round(float(pares.sim.median()), 3)}
print(json.dumps(resumen, indent=1, ensure_ascii=False))

{
 "pares_vecinos": 98330,
 "vecinos_comparten_entidad_pct": 52.4,
 "azar_comparten_entidad_pct": 2.77,
 "vecinos_compitieron_pct": 38.7,
 "azar_compitieron_pct": 1.36,
 "similitud_mediana_vecinos": 0.312
}


### Sustitutos por categoría
La pregunta pide sustitutos "por entidad o **categoría**". Se separa el resultado según la categoría principal del proveedor (la que más aparece en sus participaciones): bienes, servicios u obras. Así se ve si la similitud de contenido identifica sustitutos en todos los rubros o solo en alguno.

In [11]:
cat_prov = pd.read_parquet(f"{RUTA_P2}/features_proveedor")[["proveedor_idx", "categoria_principal"]].set_index("proveedor_idx").categoria_principal
pares["categoria"] = pares.a.map(lambda a: cat_prov.get(prov_ids[a], "sin dato")).fillna("sin dato")
nombres = {"goods": "bienes", "services": "servicios", "works": "obras", "sin dato": "sin dato"}
por_cat = (pares.groupby("categoria").agg(pares=("a", "size"), similitud_mediana=("sim", "median"),
           comparten_entidad_pct=("ent_comun", lambda x: 100 * (x > 0).mean()), compitieron_pct=("peso", lambda x: 100 * (x > 0).mean())).round(2).reset_index())
por_cat["categoria"] = por_cat.categoria.map(nombres)
por_cat.to_csv(f"{WEB}/sustitutos_categoria.csv", index=False)
print(por_cat.to_string(index=False))

categoria  pares  similitud_mediana  comparten_entidad_pct  compitieron_pct
   bienes  51680               0.32              58.650002            41.47
servicios  38820               0.31              46.509998            37.10
    obras   7830               0.28              40.910000            28.05


In [12]:
# Los 150 proveedores con más entidades en train, con sus 5 sustitutos más probables (para elegir en la web)
n_ent_prov = np.asarray(R_full.sum(0)).ravel()
top_prov = np.argsort(-n_ent_prov)[:N_SUSTITUTOS_PROV]
pares_idx = pares.set_index(["a", "b"])
lista = []
for a in top_prov:
    vecinos = []
    for b, s in zip(vec10[a][:N_VECINOS_SUST], sim10[a][:N_VECINOS_SUST]):
        r = pares_idx.loc[(a, b)]; r = r.iloc[0] if isinstance(r, pd.DataFrame) else r
        vecinos.append({"nombre": dim_prov.nombre[prov_ids[b]], "similitud": round(float(s), 3), "entidades_en_comun": int(r.ent_comun),
                        "procesos_en_comun": int(r.peso), "entidades": int(n_ent_prov[b]), "palabras_clave": palabras[prov_ids[b]]})
    lista.append({"proveedor": dim_prov.nombre[prov_ids[a]], "entidades": int(n_ent_prov[a]), "palabras_clave": palabras[prov_ids[a]], "sustitutos": vecinos})
with open(f"{WEB}/sustitutos.json", "w", encoding="utf-8") as f:
    json.dump({"meta": resumen, "proveedores": lista}, f, ensure_ascii=False, indent=1)
x = lista[0]; print(x["proveedor"], "| entidades:", x["entidades"], "|", x["palabras_clave"])
for s in x["sustitutos"][:3]: print("  →", s["nombre"], s["similitud"], "ent. comunes:", s["entidades_en_comun"], "procesos comunes:", s["procesos_en_comun"])

GRUPO SANTA FE SOCIEDAD ANONIMA CERRADA - GRUPO SANTA FE S.A.C. | entidades: 242 | tipo ip, ip, ip 42, portland, cemento portland
  → CORPORACION HELEO SOCIEDAD ANONIMA CERRADA 0.439 ent. comunes: 18 procesos comunes: 406
  → CORPORACION CLABE S.R.L. 0.404 ent. comunes: 35 procesos comunes: 623
  → Persona natural (RUC 10424****30) 0.377 ent. comunes: 3 procesos comunes: 33


## 6. Afinidades entidad–proveedor inusuales
**Afinidad** de un par (entidad, proveedor) = coseno entre lo que **vende ese proveedor** y lo que venden **todos los demás proveedores de esa entidad** (el vector del proveedor contra la suma de los otros). 1 = vende lo mismo que el resto; 0 = nada que ver.

Un par es **inusual** si su afinidad está entre el `PCT_INUSUAL`% más bajo de todos los pares (entidades con al menos 5 proveedores, para que "el resto" tenga sentido). Entre los inusuales se ordena por **monto adjudicado** para revisar primero los más grandes.

**Límites:** (1) el vector del proveedor incluye los procesos que ganó en esa misma entidad, así que la afinidad tiende a salir *más alta* de lo real; los bajos son entonces más llamativos. (2) Textos cortos o genéricos dan afinidad baja sin que haya nada raro. (3) Es una señal para revisar, no una conclusión.

In [13]:
grado_e = np.asarray(R_full.sum(1)).ravel()
tr = train[grado_e[train.e.values] >= 5].copy()
suma_ent = R_full @ V                                  # suma de los vectores de los proveedores de cada entidad
rest = suma_ent[tr.e.values] - V[tr.p.values]          # "el resto" de la entidad, sin este proveedor
tr["afinidad"] = (normalize(rest) * V[tr.p.values]).sum(1)
tr["pct"] = tr.afinidad.rank(pct=True) * 100
print("pares evaluados:", len(tr), "| afinidad mediana:", round(tr.afinidad.median(), 3), "| umbral inusual (percentil", PCT_INUSUAL, "):", round(tr.afinidad.quantile(PCT_INUSUAL / 100), 3))

inus = tr[tr.pct <= PCT_INUSUAL].sort_values("monto_total", ascending=False)
print("pares inusuales:", len(inus), "| entidades distintas:", inus.e.nunique(), "| proveedores distintos:", inus.p.nunique())

n_ent_prov_s = pd.Series(n_ent_prov, index=np.arange(NP))
top_por_ent = train.sort_values("n_adjudicaciones", ascending=False).groupby("e").head(3).groupby("e").p.apply(list)
out = inus.head(200).copy()
out["entidad"] = [dim_ent.nombre[ent_ids[e]] for e in out.e]; out["tipo_entidad"] = [dim_ent.entidad_tipo[ent_ids[e]] for e in out.e]
out["proveedor"] = [dim_prov.nombre[prov_ids[p]] for p in out.p]
out["entidades_del_proveedor"] = n_ent_prov_s[out.p.values].astype(int).values
out["proveedores_de_la_entidad"] = grado_e[out.e.values].astype(int)
out["que_vende_el_proveedor"] = [palabras[prov_ids[p]] for p in out.p]
out["que_compra_la_entidad"] = [" | ".join(palabras[prov_ids[q]] for q in top_por_ent[e][:2]) for e in out.e]
out = out.rename(columns={"n_adjudicaciones": "n_adjudicaciones"})
cols = ["entidad", "tipo_entidad", "proveedor", "afinidad", "pct", "n_adjudicaciones", "monto_total", "entidades_del_proveedor",
        "proveedores_de_la_entidad", "que_vende_el_proveedor", "que_compra_la_entidad"]
out = out[cols].round({"afinidad": 3, "pct": 2, "monto_total": 0}); out.to_csv(f"{WEB}/afinidades_inusuales.csv", index=False)
print(out[["entidad", "proveedor", "afinidad", "monto_total"]].head(5).to_string(index=False))

pares evaluados: 42163 | afinidad mediana: 0.256 | umbral inusual (percentil 2 ): 0.06
pares inusuales: 843 | entidades distintas: 494 | proveedores distintos: 672
                                                                   entidad                                         proveedor  afinidad  monto_total
                                                   SEGURO SOCIAL DE  SALUD                             VIETTEL  PERU  S.A.C.     0.057   53815556.0
               SERVICIO DE AGUA POTABLE Y ALCANTARILLADO DE LIMA - SEDAPAL                                      QUIMPAC S.A.     0.052   53277649.0
                                                   SEGURO SOCIAL DE  SALUD INFORMACION LOCALIZADA S.A.S. - SUCURSAL DEL PERU     0.052   29511300.0
COMISION DE PROMOCION DEL PERU PARA LA EXPORTACION Y EL TURISMO - PROMPERU           PROMOTORA DE TURISMO NUEVO MUNDO S.A.C.     0.012   24688430.0
                                     SOCIEDAD ELECTRICA DEL SUR OESTE S.A.                      

## 7. Ejemplos de recomendaciones con el híbrido

In [14]:
nombre_e = lambda e: dim_ent.nombre[ent_ids[e]]
nombre_p = lambda p: dim_prov.nombre[prov_ids[p]]
S = S_hib.copy(); S[R_full.nonzero()] = -np.inf
nuevos = test.groupby("e").p.apply(set)
cand = [e for e in nuevos.index if 5 <= grado_e[e] <= 60]
hit10 = [e for e in cand if len(set(np.argsort(-S[e])[:10]) & nuevos[e]) > 0]
print(f"entidades con al menos un acierto en el top-10: {len(hit10)} de {len(cand)} evaluables (5–60 proveedores)")
elegidas = np.random.RandomState(SEED).choice(hit10, 12, replace=False)
ej = []
for e in elegidas:
    tiene = train[train.e == e].sort_values("n_adjudicaciones", ascending=False).head(5)
    top = np.argsort(-S[e])[:10]
    ej.append({"entidad": nombre_e(e), "tipo": dim_ent.entidad_tipo[ent_ids[e]], "departamento": dim_ent.entidad_departamento[ent_ids[e]],
               "proveedores_en_train": int(grado_e[e]), "contrataciones_nuevas_en_test": int(len(nuevos[e])),
               "proveedores_actuales": [{"nombre": nombre_p(p), "n_adjudicaciones": int(n), "palabras_clave": palabras[prov_ids[p]]} for p, n in zip(tiene.p, tiene.n_adjudicaciones)],
               "recomendados": [{"nombre": nombre_p(p), "palabras_clave": palabras[prov_ids[p]], "acierto": bool(p in nuevos[e])} for p in top]})
with open(f"{WEB}/ejemplos_recomendaciones.json", "w", encoding="utf-8") as f:
    json.dump({"meta": {"alfa": ALFA, "k": 10, "entidades_con_acierto": len(hit10), "entidades_evaluables": len(cand)}, "ejemplos": ej}, f, ensure_ascii=False, indent=1)
print(ej[0]["entidad"], "|", [(r["nombre"], r["acierto"]) for r in ej[0]["recomendados"][:4]])

entidades con al menos un acierto en el top-10: 236 de 976 evaluables (5–60 proveedores)
MUNICIPALIDAD PROVINCIAL DE UTCUBAMBA - BAGUA GRANDE | [('FACOIMPEC E.I.R.L.', False), ('PROCESADORA Y AGROINDUSTRIA ESPERANZA E.I.R.L.', True), ('CORPORACIÓN DUPINIS EMPRESA INDIVIDUAL DE RESPOSABILIDAD LIMITADA', False), ('GRUPO ALIMENTARIO AMAZONICO EIRL', False)]


## 8. Lectura y respuesta a P2 (con las cifras de esta corrida)

**Pregunta P2:** *¿Qué proveedores son sustitutos entre sí para una misma entidad o categoría y qué afinidades entidad–proveedor son inusuales?*

### 8.1 ¿Qué tan bien se adivina con quién contratará una entidad?
Test: **4,926 pares nuevos** de 1,331 entidades. De los 10,303 pares del test, 5,374 ya existían en train (la entidad ya había contratado a ese proveedor, no hay nada que «descubrir») y 3 pertenecen a 2 entidades que quedaron sin historia en train al aplicar los dos filtros del universo (**arranque en frío**: un recomendador colaborativo no puede recomendarles). El notebook 1 cuenta 4,929 pares nuevos porque incluye esos 3.

| Modelo | Recall@10 (intervalo 95%) | Veces mejor que "populares" (recall) |
|---|---|---|
| Populares (referencia) | 0.022 (0.017 a 0.029) | 1.0 |
| Contenido (HNSW) | 0.080 (0.068 a 0.092) | 3.6 |
| Vecindario item–item (K=800) | 0.083 (0.071 a 0.095) | 3.7 |
| Vecindario user–user (K=50) | 0.119 (0.105 a 0.132) | 5.3 |
| **ALS (128 factores)** | **0.136 (0.121 a 0.151)** | **6.1** |
| Híbrido con α=0.1 (elegido en validación) | 0.111 (0.097 a 0.124) | 5.0 |

El intervalo sale de remuestrear las entidades 1,000 veces (sección 4b). Los intervalos de varios modelos se solapan, por eso se miran también las **diferencias** entre pares de modelos:

| Comparación (recall@10) | Diferencia | Intervalo 95% | ¿Distinguible? |
|---|---|---|---|
| user–user − item–item | +0.036 | 0.024 a 0.049 | Sí |
| ALS − user–user | +0.017 | 0.006 a 0.028 | Sí, por poco |
| ALS − híbrido (α=0.1) | +0.024 | 0.013 a 0.037 | Sí |
| híbrido (α=0.1) − contenido | +0.031 | 0.023 a 0.039 | Sí |
| item–item − contenido | +0.003 | −0.010 a 0.016 | **No** |
| ALS − populares | +0.113 | 0.097 a 0.129 | Sí |

**Lectura:** ALS, user–user y el híbrido están por encima de item–item y contenido de forma creíble; item–item y contenido **no se distinguen entre sí**. ALS gana a user–user, pero con margen pequeño. ALS entrenado con 5 semillas distintas da recall@10 de 0.1366 en promedio (desviación 0.0019, entre 0.1341 y 0.1393): el resultado no depende de la semilla.

- **ALS es el mejor modelo único:** de cada 100 proveedores nuevos que una entidad contrató, ~14 estaban en su top-10 (frente a ~2 con "populares").
- **El vecindario entre entidades (user–user) supera al de proveedores (item–item):** 0.119 frente a 0.083, y la diferencia es creíble. En clase suele pasar al revés (item–item funciona mejor). La validación mostró lo mismo (0.125 frente a 0.073), así que no es un resultado del test. **Se probó una explicación y salió distinta de lo esperado** (sección 4c). La idea era que item–item sufre porque cada proveedor atiende a pocas entidades (la mediana es 3, y 46% tiene 2 o menos). Si fuera así, item–item debería quedar peor justo con esos proveedores. Pasa lo contrario:

| Proveedor a adivinar, según sus entidades en train | Pares nuevos | item–item | user–user | ALS |
|---|---|---|---|---|
| 2 o menos | 915 | **0.068** | 0.013 | 0.031 |
| 3 a 9 | 2,374 | 0.058 | 0.051 | **0.093** |
| 10 o más | 1,637 | 0.110 | **0.250** | 0.221 |

  User–user gana porque acierta con los proveedores **grandes y conocidos** (con 10 o más entidades llega a 0.250, incluso sobre ALS), y es muy débil con los pequeños. Va de la mano con la popularidad: **46% de su top-10 son los 100 proveedores más populares** (en ALS es 19%, en item–item 12%, en contenido 6%). Item–item y ALS recuperan mejor a los proveedores chicos. Por eso ALS gana en el total: es el único que rinde bien en los tres grupos.
- **Solo contenido ya triplica a "populares"**, sin usar ningún dato de quién contrata a quién.
- **La similitud elegida no cambia mucho:** en la validación coseno, Jaccard y Pearson dan resultados casi iguales en user–user (0.125, 0.124 y 0.124) y coseno es la mejor en item–item (0.073 contra 0.073 y 0.069).
- **El híbrido no ganó con el peso elegido.** Se probaron **tres cortes de validación** (70/30, 80/20 y 90/10) y los tres eligen el mismo α = 0.1 (recall@10 promedio 0.203), así que la elección es firme *dentro de train*. Pero en el test lo mejor está en α≈0.6 (recall 0.152, +0.016 sobre ALS solo). Elegir α mirando el test sería hacer trampa, por eso no se hizo. **Por qué difieren (comprobado en la sección 3):** los proveedores que hay que adivinar en la validación tienen **menos historia** que los del test: mediana de 4 entidades frente a 6, y 39.6% con 2 o menos frente a 18.6%. El contenido no necesita historia, así que ayuda más en la validación; ALS sí la necesita y rinde mejor en el test. Es una explicación coherente con los números, no una prueba. Queda como **señal de que mezclar puede ayudar un poco con proveedores que tienen historia**, pero esa mejora no se confirmó con un criterio válido; una validación que se parezca más al test podría elegir otro α.
- **Cobertura y popularidad (top-10 de las 1,331 entidades):** «populares» recomienda solo 19 proveedores distintos (0.2% de los 9,833). Contenido llega a 3,300 (33.6%), item–item a 4,767 (48.5%), ALS a 3,234 (32.9%) y el híbrido a 3,343 (34.0%). **User–user es el más concentrado entre los modelos personalizados**: 1,936 proveedores distintos (19.7%). Acertar más no significa recomendar con más variedad.
- **Por tamaño de entidad (recall@10):** ALS gana en los tres grupos: pocas (≤9 proveedores) 0.171, medias (10–26) 0.142, muchas (>26) 0.091. User–user queda segundo (0.144, 0.131 y 0.081). Contenido supera a "populares" en los tres, incluso en entidades con poca historia (0.111 frente a 0.017). Con muchos proveedores es más difícil acertar porque hay más donde elegir.
- **Los parámetros ganadores no quedaron en el borde de la rejilla.** Se ampliaron las rejillas: ALS probó hasta 256 factores y regularización 10 (256 factores rinden menos y regularización 10 destruye el modelo), y alpha 1, 10 y 40 (gana 10). Item–item probó hasta 1,600 vecinos (gana 800) y user–user probó de 5 a 800 (gana 50).

### 8.2 Proveedores sustitutos
Para cada proveedor se tomaron sus 10 vecinos por contenido (98,330 pares). **El 52% de esos pares atiende a al menos una entidad en común** (azar: 2.8%) y **el 39% se presentó a los mismos procesos** (azar: 1.4%). Es decir, "vender cosas parecidas" sí identifica competidores reales, ~19 y ~28 veces más que al azar.
- Seguros: MAPFRE ↔ La Positiva (similitud 0.59, 57 entidades en común, 557 procesos en común), Pacífico (0.56) y Rímac (0.54).
- Telefonía: América Móvil ↔ Entel (0.31; 36 entidades y 299 procesos en común).
- Los sustitutos de un proveedor, con sus entidades y procesos en común, están en `sustitutos.json` (150 proveedores con más entidades).

**Por categoría principal del proveedor:** en bienes el 58.7% de los pares comparte entidad (51,680 pares), en servicios el 46.5% (38,820) y en obras el 40.9% (7,830). Se presentan a los mismos procesos en el 41.5%, 37.1% y 28.1%. En los tres rubros es muy superior al azar (2.8% y 1.4%); es más marcado en bienes (posible razón, no verificada: son productos más estandarizados).

**Cómo leerlo:** son sustitutos *funcionales* (la entidad podría haber elegido a cualquiera de los dos), no un indicio de acuerdo entre ellos. Competir en muchos procesos y atender a las mismas entidades es lo esperable en un rubro con pocos oferentes.

### 8.3 Afinidades inusuales
De 42,163 pares (entidades con ≥5 proveedores), la afinidad mediana es 0.256; **843 pares (2%) tienen afinidad ≤0.06**, en 494 entidades. Se guardan los 200 de mayor monto.
- Lo inusual por texto casi siempre son **servicios de apoyo en entidades que compran otras cosas**: internet y telefonía (Viettel y EsSalud, S/ 53.8 millones), Google Workspace (S/ 29.5 millones), vigilancia, limpieza, pasajes (PROMPERÚ), cloro para SEDAPAL (S/ 53.3 millones; los insumos químicos para tratar agua son plausibles aunque difieran de sus otros proveedores).
- Son pares **raros para el modelo, no necesariamente raros para el negocio**. Se usan para decidir qué mirar primero, no para concluir nada.

### 8.4 Límites
1. Solo se usan los procesos *ganados* con texto; un proveedor con pocos procesos (la mediana es 4) tiene un perfil poco estable.
2. Los consorcios (RUC de 7 dígitos) están fuera: sus relaciones con las entidades no aparecen.
3. El test cubre pocos meses (desde 2026-03-01); la métrica puede cambiar con más datos. Es una sola partición temporal: los intervalos miden el azar al elegir entidades, no el cambio de periodo.
4. La afinidad incluye los procesos del propio par, así que tiende a subestimar lo inusual.
5. Arranque en frío: entidades sin historia en train no se pueden evaluar ni recomendar (2 entidades en el test).
6. Todo es una **señal descriptiva para revisar**; no prueba irregularidades.